In [2]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles:")
print(os.listdir())

Current folder:
C:\Users\shivani\Desktop\Face_project\Faces

Files:
['.ipynb_checkpoints', 'faceamma.jpg', 'faceanish.jpg', 'facedaddy.jpg', 'faceshivani.jpg', 'trainer', 'Untitled.ipynb', 'Untitled1.ipynb']


In [3]:
import cv2
import numpy as np
from insightface.app import FaceAnalysis

app = FaceAnalysis(
    name='buffalo_l',
    providers=['CPUExecutionProvider']
)

app.prepare(ctx_id=-1)

known_faces = {}

image_files = [
    "faceanish.jpg",
    "faceshivani.jpg",
    "faceamma.jpg",
    "facedaddy.jpg"
]

for image_file in image_files:

    img = cv2.imread(image_file)

    if img is None:
        print("Could not read:", image_file)
        continue

    faces = app.get(img)

    if len(faces) == 0:
        print("No face found:", image_file)
        continue

    name = image_file.replace(".jpg", "").replace("face", "")

    embedding = faces[0].embedding
    embedding = embedding / np.linalg.norm(embedding)

    known_faces[name] = embedding

    print("Added:", name)

np.save("known_faces.npy", known_faces)

print("Training Complete")

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\shivani/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\shivani/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\shivani/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\shivani/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\shivani/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112,

In [4]:
import os

print(os.path.exists("known_faces.npy"))
print(os.listdir())

True
['.ipynb_checkpoints', 'faceamma.jpg', 'faceanish.jpg', 'facedaddy.jpg', 'faceshivani.jpg', 'known_faces.npy', 'trainer', 'Untitled.ipynb', 'Untitled1.ipynb']


In [8]:
import cv2
import numpy as np
from insightface.app import FaceAnalysis

# Load InsightFace
app = FaceAnalysis(
    name='buffalo_l',
    providers=['CPUExecutionProvider']
)

app.prepare(ctx_id=-1)

# Load trained faces
known_faces = np.load(
    "known_faces.npy",
    allow_pickle=True
).item()

print("Known people:", list(known_faces.keys()))

cap = cv2.VideoCapture(0)

THRESHOLD = 0.40

while True:

    ret, frame = cap.read()

    if not ret:
        break

    faces = app.get(frame)

    for face in faces:

        embedding = face.embedding
        embedding = embedding / np.linalg.norm(embedding)

        best_name = "Unknown"
        best_score = -1

        for name, known_embedding in known_faces.items():

            score = np.dot(embedding, known_embedding)

            if score > best_score:
                best_score = score
                best_name = name

        if best_score < THRESHOLD:
            best_name = "Unknown"

        x1, y1, x2, y2 = face.bbox.astype(int)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        cv2.putText(
            frame,
            f"{best_name} ({best_score:.2f})",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

    cv2.imshow("Face Recognition", frame)

    if cv2.waitKey(1) & 0xFF == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\shivani/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\shivani/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\shivani/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\shivani/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\shivani/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112,

KeyboardInterrupt: 

In [ ]:
cv2.destroyAllWindows()